Consideraciones Académicas y Simplificaciones Didácticas

Este notebook implementa una arquitectura Transformer Encoder-Decoder funcional basada en el paper "Attention Is All You Need". Sin embargo, con el objetivo de mantener el código legible y enfocado en el flujo de datos (tensores), se han realizado ciertas simplificaciones respecto a una implementación de producción (SOTA).

Si estás estudiando el paper en profundidad, ten en cuenta las siguientes observaciones críticas:

1. Escalamiento de Embeddings ($\sqrt{d_{model}}$)

 - Observación: En este código, sumamos directamente Embedding + PositionalEncoding.
 - Lo que dice el paper: El paper original multiplica los embeddings por $\sqrt{d_{model}}$ antes de sumar la posición.
 - ¿Por qué? Esto equilibra la varianza de los embeddings con la del Positional Encoding (que tiene valores fijos entre -1 y 1). Sin este escalado, en modelos profundos, la información de posición podría dominar la semántica al inicio.
 - "Entonces ¿Por qué mi loss oscila al principio?" -> En modelos pequeños (mini) no es crítico, pero es una diferencia matemática importante.

 2. La "Caja Negra" nn.Transformer
 - Observación: Usamos el módulo de alto nivel de PyTorch.
 - ¿Dónde ocurre la multiplicación $QK^T$ o la división por $\sqrt{d_k}$? ->  Esas operaciones ocurren dentro de nn.Transformer. Este script se enfoca en la arquitectura macro (cómo viajan los datos entre Encoder y Decoder) y no en la micro-matemática de la atención.

 3. Restricción de Dimensiones (d_model vs nhead)
 - Observación: Configuramos d_model=48 y nhead=4.
 - Advertencia: Si cambias estos valores, asegúrate siempre de que d_model sea divisible por nhead (ej. 50 / 4 no es entero). Si no lo es, PyTorch lanzará un error de dimensión. Este es pequeño por ello limita su capacidad para tareas complejas reales, donde se requieren tamaños mayores (e.g., d_model=512+).

 4. Eficiencia en Inferencia (KV-Cache)
 - Observación: La función greedy_decode vuelve a procesar toda la frase cada vez que genera una palabra nueva ("hola" -> "hola mundo" -> "hola mundo cruel").

 - Entonces"¿Así funciona ChatGPT?" -> No. En producción se utiliza KV-Caching para guardar los cálculos pasados y no repetirlos. Aquí usamos el método ineficiente ($O(N^2)$) porque es mucho más fácil de entender y codificar para fines educativos.

 5. Tokenización Básica
 - Observación: Usamos .split() por espacios.
 - Limitación: El modelo no entiende que "hola," (con coma) es lo mismo que "hola" (sin coma). En un sistema real se usan tokenizadores de sub-palabras (BPE, WordPiece) para manejar puntuación y morfología.

No se incluye label smoothing para estabilizar el entrenamiento.

Extensiones

 - Mejoras en Inferencia: Implementar beam search o top-k sampling para generaciones más diversas.
 - Tokenización Avanzada: Integrar BPE/WordPiece (e.g., via HuggingFace) para manejar vocabularios grandes y sub-palabras.
 - Escalado: Aumentar parámetros y usar datasets reales (e.g., WMT para traducción) con GPUs para pre-entrenamiento.
 - Aplicaciones: Adaptar a encoder-only (BERT-like) o decoder-only (GPT-like); extender a visión (ViT) o multimodal (e.g., CLIP).
 - Optimizaciones: Agregar Noam scheduler, dropout en attention, o eficiencia (e.g., torch.compile).

In [ ]:
# -----------------------------------------------------------------------------
# OBJETIVO ACADÉMICO
# -----------------------------------------------------------------------------
# Este script implementa un Transformer Encoder–Decoder REAL (paper: "Attention Is All You Need")
# usando nn.Transformer de PyTorch, pero en tamaño "mini" para poder explicarlo.
#
# La idea es que puedas mapear 1 a 1 con el diagrama clásico:
#
#   Inputs -> [Input Embedding] ⊕ [Positional Encoding] -> (Encoder N×) -> memory
#   Outputs shifted right -> [Output Embedding] ⊕ [Positional Encoding] -> (Decoder N×)
#   -> Linear -> Softmax -> Output Probabilities
#
# Donde N× significa repetir un bloque varias veces.
# Tener en cuenta que memory es la forma de pasar de Encoder N× a ecoder N× ya que no figura como caja en proceso
# Dentro de cada bloque del paper ocurren estas cajas:
#
# EncoderLayer (por capa):
#   1) Multi-Head Self-Attention
#   2) Add & Norm (residual + LayerNorm)
#   3) Feed Forward (FFN)
#   4) Add & Norm
#
# DecoderLayer (por capa):
#   1) Masked Multi-Head Self-Attention (causal: no mira el futuro)
#   2) Add & Norm
#   3) Multi-Head Cross-Attention (mira la "memory" del encoder)
#   4) Add & Norm
#   5) Feed Forward (FFN)
#   6) Add & Norm
#
# NOTA: nn.Transformer encapsula esas cajas internamente. Aquí las "vemos" por
#       las entradas que le pasamos: embeddings, positional encoding, máscaras, etc.
#
# Requisitos:
# pip install torch
# Autor: Miguel Angel cotrina Espinoza
# -----------------------------------------------------------------------------

In [2]:
import math
import random
from dataclasses import dataclass
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import warnings
warnings.filterwarnings("ignore")

In [3]:
## Libreria auxiliar de calculo de parametros
def count_parameters(model: nn.Module):
    total_params = 0
    trainable_params = 0

    print("\n📊 Conteo de parámetros por componente:\n")

    for name, param in model.named_parameters():
        num = param.numel()
        total_params += num
        if param.requires_grad:
            trainable_params += num

        print(f"{name:50s} | {num:8d} parámetros")

    print("\n" + "-" * 70)
    print(f"🔢 Total de parámetros        : {total_params:,}")
    print(f"🧠 Parámetros entrenables     : {trainable_params:,}")
    print("-" * 70)

    return total_params, trainable_params

In [4]:
# -----------------------------------------------------------------------------
# 1) CONFIGURACIÓN: Define "tamaño" del modelo (mini pero funcional)
# -----------------------------------------------------------------------------
@dataclass
class CFG:
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Dimensión del embedding principal (d_model del paper)
    d_model: int = 48 # 32

    # Número de cabezas (h del paper, Multi-Head Attention)
    nhead: int = 4 # 2

    # N× del Encoder y del Decoder
    num_encoder_layers: int = 2
    num_decoder_layers: int = 2

    # Dimensión interna del Feed Forward (d_ff del paper)
    dim_feedforward: int = 128 #64

    dropout: float = 0.1

    # Longitud máxima de secuencia (para padding y positional encoding)
    max_len: int = 20 #16

    # Entrenamiento
    batch_size: int = 64 # 32
    lr: float = 1e-3 # 2e-3
    epochs: int = 30 #25 # con 7k ejemplos converge bien

    # Inferencia (generación token a token)
    max_gen_len: int = 10 # respuestas cortas
    temperature: float = 0.8 # evita repeticiones

cfg = CFG()
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

## Tabla de manejo de parametros

| Parámetro      | Impacto | Comentario                  |
|----------------|---------|-----------------------------|
| d_model        | ⭐⭐⭐  | El más importante           |
| num_layers     | ⭐⭐   | Más profundidad             |
| dim_feedforward| ⭐⭐   | Más capacidad               |
| vocab_size     | ⭐     | Impacta embeddings          |
| nhead          | ⚠️     | No cambia params totales mucho |
 .

In [5]:
# -----------------------------------------------------------------------------
# 2) DATASET: Pares (Input -> Output) en lenguaje natural muy básico
# -----------------------------------------------------------------------------
# Esto es intencionalmente pequeño: el objetivo es mostrar que el modelo aprende
# un micro-dominio en español y que el flujo se ve "vivo".

def load_pairs_from_txt(path: str) -> List[Tuple[str, str]]:
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            x, y = line.split("\t")
            pairs.append((x, y))
    return pairs

DATASET_PATH = "/content/dataset_bigmodelmac7000.txt"
PAIRS = load_pairs_from_txt(DATASET_PATH)


# Si quieres "más natural", amplia tus variaciones:
# PAIRS += [("hola amigo", "hola"), ("que tal", "bien"), ...]

In [6]:
# -----------------------------------------------------------------------------
# 3) TOKENIZACIÓN + VOCABULARIO
# -----------------------------------------------------------------------------
# En un sistema real usarías BPE/WordPiece, pero aquí un tokenizador simple

# Cada palabra separada por espacio = 1 token
# No hay sub-palabras
# No hay compresión del vocabulario

# Métodos como BPE (Byte Pair Encoding) o WordPiece:
# Dividen palabras en sub-palabras frecuentes
# Ejemplo:
#   feliz      → fe + liz
#   felicidad → fe + lic + idad

# es suficiente para explicar, por motivos pedagógicos pero no en produccion.
SPECIALS = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
PAD, UNK, BOS, EOS = SPECIALS

# PAD Cuando usamos batches, todas las secuencias deben tener el mismo largo, lo usamos para llenar los espacio/token.
# UNK Palabras fuera del vocabulario
# BOS Marca el inicio del texto que el decoder debe generar.
# EOS Marca el final del texto.

def basic_tokenize(text: str) -> List[str]:
    # tokenización mínima: minúsculas + split por espacios
    return text.strip().lower().split()

def build_vocab(pairs: List[Tuple[str, str]]) -> Tuple[Dict[str, int], Dict[int, str]]:
    tokens = []
    for x, y in pairs:
        tokens.extend(basic_tokenize(x))
        tokens.extend(basic_tokenize(y))
    uniq = SPECIALS + sorted(set(tokens))
    stoi = {t: i for i, t in enumerate(uniq)}  # string->id
    itos = {i: t for t, i in stoi.items()}     # id->string
    return stoi, itos

stoi, itos = build_vocab(PAIRS)
vocab_size = len(stoi)

def encode(tokens: List[str]) -> List[int]:
    return [stoi.get(t, stoi[UNK]) for t in tokens]

def decode(ids: List[int]) -> str:
    # decodifica ids a texto, cortando al encontrar <EOS>
    toks = []
    for i in ids:
        t = itos.get(i, UNK)
        if t in (PAD, BOS):
            continue
        if t == EOS:
            break
        toks.append(t)
    return " ".join(toks)

In [7]:
# -----------------------------------------------------------------------------
# 4) CONSTRUCCIÓN DE EJEMPLOS: Esto mapea directamente a:
#    - Inputs (Encoder)
#    - Outputs (shifted right) (Decoder input)
#    - Targets (lo que debe predecir el decoder)
# -----------------------------------------------------------------------------
def make_example(x_text: str, y_text: str):
    # Recortamos para no exceder max_len
    x_ids = encode(basic_tokenize(x_text))[: cfg.max_len - 2]
    y_ids = encode(basic_tokenize(y_text))[: cfg.max_len - 2]

    # ENCODER INPUT (Inputs en diagrama):
    # src = [tokens_input..., <EOS>]
    src = torch.tensor(x_ids + [stoi[EOS]], dtype=torch.long)

    # DECODER INPUT (Outputs shifted right en diagrama):
    # tgt_in = [<BOS>, tokens_output...]
    # Esto implementa teacher forcing: el decoder recibe la "respuesta correcta"
    # pero desplazada; así aprende a predecir el siguiente token.
    tgt_in = torch.tensor([stoi[BOS]] + y_ids, dtype=torch.long)

    # TARGET (labels): lo que se compara contra la predicción
    # tgt_out = [tokens_output..., <EOS>]
    tgt_out = torch.tensor(y_ids + [stoi[EOS]], dtype=torch.long)

    return src, tgt_in, tgt_out

In [8]:
# -----------------------------------------------------------------------------
# 5) COLLATE (batch): padding y masks
# -----------------------------------------------------------------------------
# En el diagrama no se dibuja el padding, pero en práctica es crucial:
# - PAD agrega tokens "vacíos" para que todos en el batch tengan mismo largo.
# - src_key_padding_mask y tgt_key_padding_mask indican qué posiciones son PAD
#   para que attention NO les preste atención.
def collate(batch):
    srcs, tgts_in, tgts_out = zip(*batch)

    srcs = pad_sequence(srcs, batch_first=True, padding_value=stoi[PAD])
    tgts_in = pad_sequence(tgts_in, batch_first=True, padding_value=stoi[PAD])
    tgts_out = pad_sequence(tgts_out, batch_first=True, padding_value=stoi[PAD])

    src_key_padding_mask = (srcs == stoi[PAD])     # True donde hay PAD
    tgt_key_padding_mask = (tgts_in == stoi[PAD])  # True donde hay PAD

    return srcs, tgts_in, tgts_out, src_key_padding_mask, tgt_key_padding_mask

def make_batches(pairs, batch_size):
    data = [make_example(x, y) for x, y in pairs]
    random.shuffle(data)
    for i in range(0, len(data), batch_size):
        yield collate(data[i:i + batch_size])


In [9]:
# -----------------------------------------------------------------------------
# 6) POSITIONAL ENCODING (caja "Positional Encoding" del paper)
# -----------------------------------------------------------------------------
# El Transformer no es secuencial como RNN, por eso requiere agregar posición.
# Implementamos el positional encoding sinusoidal del paper.
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float, max_len: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # pe[t, i] define la componente i de la posición t
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model) para broadcasting en batch
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (B, T, d_model)
        # Esto corresponde a la suma ⊕ del diagrama:
        # Embedding ⊕ PositionalEncoding
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


In [10]:
# -----------------------------------------------------------------------------
# 7) MODELO: Encaja 1 a 1 con el diagrama completo
# -----------------------------------------------------------------------------
class MiniTransformer(nn.Module):
    def __init__(self, vocab_size: int, cfg: CFG):
        super().__init__()
        self.cfg = cfg

        # [Input Embedding] y [Output Embedding]
        # Aquí compartimos la misma tabla de embedding para simplificar
        self.tok_emb = nn.Embedding(vocab_size, cfg.d_model)

        # [Positional Encoding]
        self.pos_enc = PositionalEncoding(cfg.d_model, cfg.dropout, max_len=cfg.max_len + 5)

        # Núcleo: Transformer Encoder–Decoder (paper)
        # - Dentro hay Encoder N× y Decoder N×
        # - Dentro de cada layer están: MHA, Add&Norm, FFN, Add&Norm, etc.
        self.transformer = nn.Transformer(
            d_model=cfg.d_model,
            nhead=cfg.nhead,
            num_encoder_layers=cfg.num_encoder_layers,  # Encoder N×
            num_decoder_layers=cfg.num_decoder_layers,  # Decoder N×
            dim_feedforward=cfg.dim_feedforward,        # FFN d_ff
            dropout=cfg.dropout,
            batch_first=True,
        )

        # [Linear] final del paper (proyección a vocabulario)
        self.out = nn.Linear(cfg.d_model, vocab_size)

    # -------------------------------------------------------------------------
    # Máscara causal (caja "Masked Multi-Head Attention" del decoder)
    # -------------------------------------------------------------------------
    def generate_square_subsequent_mask(self, sz: int) -> torch.Tensor:
        # Queremos bloquear atención a posiciones futuras:
        # para t, no puede mirar t+1, t+2, ...
        #
        # mask boolean arriba de la diagonal:
        future = torch.triu(torch.ones(sz, sz), diagonal=1).bool()

        # nn.Transformer espera una máscara float con -inf donde NO se permite
        # Atención. Softmax(-inf) -> 0.
        float_mask = torch.zeros(sz, sz)
        float_mask.masked_fill_(future, float("-inf"))
        return float_mask

    # -------------------------------------------------------------------------
    # Forward: define el flujo de flechas del diagrama
    # -------------------------------------------------------------------------
    def forward(self, src, tgt_in, src_key_padding_mask, tgt_key_padding_mask):
        # ---------------------------
        # Encoder Input path:
        # Inputs -> Input Embedding -> ⊕ PositionalEncoding
        # ---------------------------
        src_emb = self.pos_enc(self.tok_emb(src))  # (B, S, d_model)

        # ---------------------------
        # Decoder Input path:
        # Outputs shifted right -> Output Embedding -> ⊕ PositionalEncoding
        # ---------------------------
        tgt_emb = self.pos_enc(self.tok_emb(tgt_in))  # (B, T, d_model)

        # ---------------------------
        # Masked Self-Attention del decoder:
        # (bloquea futuro en atención)
        # ---------------------------
        tgt_mask = self.generate_square_subsequent_mask(tgt_in.size(1)).to(tgt_in.device)  # (T, T)

        # ---------------------------
        # Core Transformer (Encoder + Decoder)
        #
        # Internamente:
        # Encoder N×:
        #   - Multi-Head Self-Attention
        #   - Add & Norm
        #   - Feed Forward
        #   - Add & Norm
        #
        # Decoder N×:
        #   - Masked Multi-Head Self-Attention (usa tgt_mask)
        #   - Add & Norm
        #   - Cross-Attention (Queries del decoder, Keys/Values de encoder output "memory")
        #   - Add & Norm
        #   - Feed Forward
        #   - Add & Norm
        #
        # memory_key_padding_mask hace que el decoder ignore PAD del encoder.
        # ---------------------------
        h = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )  # h: (B, T, d_model) salida del decoder (antes de Linear/Softmax)

        # ---------------------------
        # Linear -> logits (antes de softmax)
        # ---------------------------
        logits = self.out(h)  # (B, T, vocab_size)
        return logits


In [11]:
# -----------------------------------------------------------------------------
# 8) ENTRENAMIENTO: CrossEntropyLoss + teacher forcing
# -----------------------------------------------------------------------------
def train():
    model = MiniTransformer(vocab_size, cfg).to(cfg.device)
    count_parameters(model)

    # Optimización (no está en el diagrama, es "cómo se entrena")
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)

    # CrossEntropyLoss implementa: LogSoftmax + NLLLoss (estable numéricamente)
    # ignore_index ignora PAD en la pérdida
    loss_fn = nn.CrossEntropyLoss(ignore_index=stoi[PAD])

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        total_loss, steps = 0.0, 0

        for src, tgt_in, tgt_out, src_kpm, tgt_kpm in make_batches(PAIRS, cfg.batch_size):
            # mover al device
            src = src.to(cfg.device)
            tgt_in = tgt_in.to(cfg.device)
            tgt_out = tgt_out.to(cfg.device)
            src_kpm = src_kpm.to(cfg.device)
            tgt_kpm = tgt_kpm.to(cfg.device)

            # Forward: produce logits para cada posición del decoder
            logits = model(src, tgt_in, src_kpm, tgt_kpm)  # (B, T, V)

            # Pérdida: comparar logits con tgt_out (targets)
            # reshape para (B*T, V) vs (B*T)
            loss = loss_fn(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item()
            steps += 1

        avg = total_loss / max(steps, 1)
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d} | loss={avg:.4f}")
            demo(model, "hola")
            demo(model, "me siento triste")
            demo(model, "como estas")

    return model


In [12]:
# -----------------------------------------------------------------------------
# 9) INFERENCIA (Generación token a token)
# -----------------------------------------------------------------------------
# Esta sección implementa la flecha "Output Probabilities" del diagrama,
# pero en modo autoregresivo:
#  - empieza con <BOS>
#  - predice el siguiente token (argmax de softmax)
#  - lo concatena al output
#  - repite hasta <EOS> o max_len
@torch.no_grad()
def greedy_decode(model, src_text: str) -> str:
    model.eval()

    # ---- Inputs ----
    # (Inputs del encoder)
    src_ids = encode(basic_tokenize(src_text))[: cfg.max_len - 1]
    src = torch.tensor(src_ids + [stoi[EOS]], dtype=torch.long).unsqueeze(0).to(cfg.device)
    src_kpm = (src == stoi[PAD])

    # ---- Outputs shifted right inicial ----
    # decoder empieza con <BOS>
    tgt = torch.tensor([[stoi[BOS]]], dtype=torch.long).to(cfg.device)

    for _ in range(cfg.max_gen_len):
        tgt_kpm = (tgt == stoi[PAD])

        # logits para cada posición del tgt actual
        logits = model(src, tgt, src_kpm, tgt_kpm)  # (1, T, V)

        # tomamos solo la última posición: "próximo token"
        next_logits = logits[:, -1, :] / max(cfg.temperature, 1e-6)

        # Softmax (Output Probabilities)
        probs = torch.softmax(next_logits, dim=-1)

        # Selección greedy (argmax) -> "token a token"
        next_id = torch.argmax(probs, dim=-1).item()

        # Agrega token al output (flecha de retroalimentación del decoder)
        tgt = torch.cat([tgt, torch.tensor([[next_id]], device=cfg.device)], dim=1)

        if next_id == stoi[EOS]:
            break

    return decode(tgt.squeeze(0).tolist())

def demo(model, text):
    out = greedy_decode(model, text)
    print(f"  IN : {text}")
    print(f"  OUT: {out}")

In [13]:
# -----------------------------------------------------------------------------
# 10) MAIN: Entrena y luego permite probar interactivo (tiempo medio 3 minutos de proceso)
# -----------------------------------------------------------------------------
print(f"Device: {cfg.device} | Vocab size: {vocab_size}")
print("Entrenando mini Transformer (Encoder–Decoder) ...")
model = train()

Device: cpu | Vocab size: 360
Entrenando mini Transformer (Encoder–Decoder) ...

📊 Conteo de parámetros por componente:

tok_emb.weight                                     |    17280 parámetros
transformer.encoder.layers.0.self_attn.in_proj_weight |     6912 parámetros
transformer.encoder.layers.0.self_attn.in_proj_bias |      144 parámetros
transformer.encoder.layers.0.self_attn.out_proj.weight |     2304 parámetros
transformer.encoder.layers.0.self_attn.out_proj.bias |       48 parámetros
transformer.encoder.layers.0.linear1.weight        |     6144 parámetros
transformer.encoder.layers.0.linear1.bias          |      128 parámetros
transformer.encoder.layers.0.linear2.weight        |     6144 parámetros
transformer.encoder.layers.0.linear2.bias          |       48 parámetros
transformer.encoder.layers.0.norm1.weight          |       48 parámetros
transformer.encoder.layers.0.norm1.bias            |       48 parámetros
transformer.encoder.layers.0.norm2.weight          |       48 pará

In [14]:
while True:
    try:
        s = input("> ")
    except EOFError:
        break

    if not s.strip():
        continue

    # Comando de salida
    if s.strip().upper() == "XEND":
        print("Saliendo del programa...")
        break

    print(greedy_decode(model, s))

> hola
hola
> hoy salio el sol
que bueno
> no me gusta el frio
que bueno
> me salio mas un proyecto
que triste
> cuentame algo
entiendo
> como estas
hola
> estoy feliz
me alegra
> XEND
Saliendo del programa...
